In [37]:
import csv, re, textwrap, yaml
from pathlib import Path
from openpyxl import load_workbook

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src').is_dir() and (p / 'output').is_dir())
BATCH = ROOT / 'output/judge_error_annotations/batch_opus_4-6_mistakes'
INSTANCES = ROOT / ('output/iclr_test_instances/pairwise_data/20260517_171954/'
                    'manipulated/iclr_test_instances.yaml')
OUT = ROOT / 'output/figures/judge-error-examples/judge_error_examples.tex'
ANNOTATOR = 'tom'
GENERATOR = 'claude-sonnet-4-5'

rows = list(load_workbook(BATCH / 'key.xlsx')['key'].iter_rows(values_only=True))
key = {r['item_id']: r for r in (dict(zip(rows[0], v)) for v in rows[1:])
       if r['annotator'] == ANNOTATOR}
with open(BATCH / 'answers' / f'answers_{ANNOTATOR}.csv', newline='') as f:
    answers = {r['item_id']: r for r in csv.DictReader(f)}
inst = yaml.safe_load(INSTANCES.read_text())

EX = {}
for i, a in answers.items():
    k = key[i]
    x = inst[int(k['problem_id'])]
    ideas = {str(n): t for n, t in x['ideas'].items()}
    meta = {str(n): m for n, m in x['metadata'].items()}
    human = str(k['gold_idea'])
    llm = next(n for n in ideas if n != human)
    # The annotator saw the pair as A/B in an order randomised per item; this is
    # the one place that is undone.
    idx = {'A': str(k['shown_A_idea']), 'B': str(k['shown_B_idea'])}
    note = a['notes'].strip()
    note = re.sub(r'\b[Ii]dea[- ]([AB])\b', lambda m: idx[m.group(1)], note)
    note = re.sub(r'(?<![A-Za-z0-9-])([AB])(?![A-Za-z0-9-])',
                  lambda m: idx[m.group(1)], note)
    EX[i] = {'paper': meta[human]['title'], 'area': x['context'],
             'ideas': sorted([(human, 'human-authored', ideas[human]),
                              (llm, f'{GENERATOR}-generated', ideas[llm])]),
             'pick': idx.get(a['your_choice'], a['your_choice']),
             'note': note}

print(len(EX), 'examples:', ' '.join(sorted(EX)))

21 examples: 001 002 003 004 005 006 007 008 009 010 011 012 013 014 015 016 017 018 019 020 021


In [38]:
def show(i, width=100):
    e = EX[i]
    wrap = lambda t: textwrap.fill(' '.join(t.split()), width)
    print(f"[{i}] {e['paper']}")
    print(f"area: {e['area']}\n")
    for idx, source, text in e['ideas']:
        print(f"{idx} ({source})\n" + wrap(text) + '\n')
    print(f"expert selected: {e['pick']}\n" + wrap(e['note']))


show('004')

[004] Debiased and Denoised Representation Learning for Incomplete Multi-view Clustering
area: unsupervised, self-supervised, semi-supervised, and supervised representation learning

0 (claude-sonnet-4-5-generated)
Representation learning paradigms—unsupervised, self-supervised, semi-supervised, and supervised—are
typically treated as distinct methodologies requiring separate architectural designs and training
protocols. We introduce Spectrum Learning, a unified framework that views supervision as a
continuous spectrum rather than discrete categories. Our approach employs a meta-learned weighting
mechanism that dynamically modulates the contribution of multiple learning objectives based on local
supervision density in the feature space. By treating each data point as existing along a
supervision gradient, the framework adaptively combines contrastive self-supervised losses,
consistency regularization, pseudo-labeling, and supervised classification within a single coherent
optimization.

In [39]:
PICKS = ['002', '004']

CAPTION = (
    r'\textbf{Judge errors with expert adjudication.} Pairs from the \humangen set on '
    r'which the judge scored the \texttt{' + GENERATOR + r'}-generated idea as more '
    r'novel than the accepted ICLR paper. Each idea is labelled with its index in the '
    r'pair and its source. The expert read the pair blind, with the two ideas in a '
    r'randomised order, and refers to them by index.'
)

ESC = {'&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#', '_': r'\_',
       '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}',
       '^': r'\textasciicircum{}',
       '<': r'\textless{}', '>': r'\textgreater{}'}


def tex(s):
    s = ''.join(ESC.get(c, c) for c in ' '.join(s.split()).replace('*', ''))
    s = s.replace('—', '---').replace('–', '--')
    return re.sub('"([^"]*)"', lambda m: '``' + m.group(1) + "''", s)


def label(top, bottom):
    return r'\textbf{' + top + r'} \newline ' + bottom


def to_latex(ids):
    body = []
    for i in ids:
        e = EX[i]
        if body:
            body += [r'\addlinespace[4pt]', r'\midrule', r'\addlinespace[2pt]']
        body += [
            r'\multicolumn{2}{@{}l}{\textit{ICLR area: ' + tex(e['area']) + r'}} \\',
            r'\addlinespace[4pt]',
        ]
        for idx, source, text in e['ideas']:
            body += [label(idx, '(' + source + ')') + ' & ' + tex(text) + r' \\',
                     r'\addlinespace[5pt]']
        body.append(label('Expert', 'selected: ' + e['pick']) + ' & '
                    + tex(e['note']) + r' \\')
    out = '\n'.join([
        r'\begin{table*}[t]', r'\centering', r'\footnotesize',
        r'\setlength{\tabcolsep}{4pt}',
        r'\renewcommand{\arraystretch}{1.2}',
        r'\begin{tabularx}{\textwidth}'
        r'{@{}>{\raggedright\arraybackslash}p{0.17\textwidth} X@{}}',
        r'\toprule',
        *body,
        r'\bottomrule', r'\end{tabularx}',
        r'\caption{' + CAPTION + '}',
        r'\label{tab:judge_error_examples}', r'\end{table*}',
    ]) + '\n'
    OUT.parent.mkdir(parents=True, exist_ok=True)
    OUT.write_text(out)
    print(out)


to_latex(PICKS)

\begin{table*}[t]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.2}
\begin{tabularx}{\textwidth}{@{}>{\raggedright\arraybackslash}p{0.17\textwidth} X@{}}
\toprule
\multicolumn{2}{@{}l}{\textit{ICLR area: applications to computer vision, audio, language, and other modalities}} \\
\addlinespace[4pt]
\textbf{0} \newline (human-authored) & Despite advances in pretraining with extended context sizes, large language models (LLMs) still face challenges in effectively utilizing real-world long-context information, primarily due to insufficient long-context alignment caused by data quality issues, training inefficiencies, and the lack of well-designed optimization objectives. To address these limitations, we propose a framework named Short-to-Long Preference Optimization (SoLoPO), decoupling long-context preference optimization (PO) into two components: short-context PO and short-to-long reward alignment (SoLo-RA), supported by both theoretical and empirical